# Digital Twin-Enabled IoT and Reinforcement Learning for Sustainable College Event Management

**End-to-end research notebook — eight stages**

This notebook downloads and processes real public datasets, then builds an event-management *scenario layer* for controlled experiments. Important provenance distinction:

- Electricity measurements come from the UCI Appliances Energy Prediction dataset (real measured household/building observations).
- Water measurements are retrieved from NYC Open Data (real city-level observations; not college-event measurements).
- Waste intensity comes from the World Bank *What a Waste 3.0* public database when its workbook is accessible. If its schema or download changes, the notebook stops with a clear message rather than silently inventing values.
- Attendance, event type, duration, minimum service levels, and event-to-campus scaling are **explicit experimental scenario inputs**, not historical observations. They are generated transparently and are never represented as real college event records.

The resulting model is therefore a reproducible *transfer/simulation study* using public resource data, not a claim that the public sources contain college-event telemetry. Set `USE_WASTE_WORKBOOK=False` only if you deliberately want to supply a separately documented public waste-intensity CSV.

Run cells from top to bottom. Internet access is required for first-time data acquisition. The notebook is designed for CPU execution; training sizes can be increased after verifying the pipeline.

## 0. Environment and reproducibility

In [ ]:
%pip -q install numpy pandas scipy scikit-learn matplotlib seaborn requests openpyxl torch paho-mqtt

In [ ]:
import os, re, json, time, math, zipfile, warnings, random, io
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from scipy.stats import wilcoxon
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("outputs")
DATA_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
print("Device:", DEVICE, "| Outputs:", OUTPUT_DIR.resolve())

## Stage 1 — Historical public dataset acquisition and feature extraction

### Sources

1. **UCI Appliances Energy Prediction** — 19,735 ten-minute observations with measured appliance electricity and indoor/outdoor conditions. DOI: `10.24432/C5VC8C` (UCI page lists DOI `10.24432/C5VC8C` / repository citation; verify the citation shown on the repository page when publishing).
2. **NYC Open Data — Water Consumption in the City of New York** — public daily city-wide water consumption. The Socrata endpoint is queried at runtime.
3. **World Bank — What a Waste 3.0** — public country/city waste-management workbook. It is used only for waste intensity and management indicators, not as event-level waste telemetry.

If a source schema changes, inspect the printed columns and adjust the mapping cell; do not substitute fabricated observations.

In [ ]:
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/374/appliances+energy+prediction.zip"
UCI_ZIP = DATA_DIR / "uci_appliances.zip"
UCI_DIR = DATA_DIR / "uci_appliances"

def download_file(url, destination, timeout=120):
    destination = Path(destination)
    if not destination.exists():
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
        destination.write_bytes(r.content)
    return destination

download_file(UCI_ZIP_URL, UCI_ZIP)
UCI_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(UCI_ZIP) as zf:
    zf.extractall(UCI_DIR)
csv_files = list(UCI_DIR.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError("UCI archive downloaded but no CSV file was found.")
energy_path = next((p for p in csv_files if "energydata_complete" in p.name.lower()), csv_files[0])
energy_raw = pd.read_csv(energy_path)
print("Energy source:", energy_path, energy_raw.shape)
display(energy_raw.head())
print(energy_raw.columns.tolist())

In [ ]:
# NYC Open Data Socrata resource. Data is city-wide and is not campus-specific.
NYC_WATER_URL = "https://data.cityofnewyork.us/resource/ia2c-pyqv.csv?$limit=50000"
water_path = DATA_DIR / "nyc_water.csv"
try:
    if not water_path.exists():
        wr = requests.get(NYC_WATER_URL, timeout=90)
        wr.raise_for_status()
        water_path.write_bytes(wr.content)
    water_raw = pd.read_csv(water_path)
    print("Water source:", water_raw.shape)
    display(water_raw.head())
    print(water_raw.columns.tolist())
except Exception as e:
    raise RuntimeError(
        "Could not retrieve the configured NYC Open Data water resource. "
        "Check the Socrata dataset identifier/network, then update NYC_WATER_URL. "
        "The notebook intentionally does not generate replacement water observations."
    ) from e

In [ ]:
# World Bank What a Waste 3.0 — public country and city workbooks.
WB_COUNTRY_XLSX = (
    "https://datacatalogfiles.worldbank.org/ddh-published/0039597/DR0095900/"
    "What_a_Waste_3.0_COUNTRY_Dataset_%26_Codebook.xlsx"
)
WB_CITY_XLSX = (
    "https://datacatalogfiles.worldbank.org/ddh-published/0039597/DR0095900/"
    "What_a_Waste_3.0_CITY_Dataset_%26_Codebook.xlsx"
)
USE_WASTE_WORKBOOK = True
waste_raw = None
if USE_WASTE_WORKBOOK:
    try:
        wb_path = download_file(WB_COUNTRY_XLSX, DATA_DIR / "world_bank_what_a_waste_country.xlsx")
        wb = pd.ExcelFile(wb_path)
        print("Waste workbook sheets:", wb.sheet_names)
        waste_frames = {s: pd.read_excel(wb_path, sheet_name=s) for s in wb.sheet_names}
        for s, frame in waste_frames.items():
            print("\n", s, frame.shape, frame.columns.tolist()[:25])
        # Select the sheet most likely to contain the country observations.
        waste_raw = max(waste_frames.values(), key=lambda x: x.shape[0] * max(x.shape[1], 1))
        display(waste_raw.head())
    except Exception as e:
        raise RuntimeError(
            "World Bank workbook could not be downloaded/read. Review the official "
            "What a Waste 3.0 catalog link and update WB_COUNTRY_XLSX. No waste values "
            "are synthesized as a substitute."
        ) from e

### Normalize source fields

In [ ]:
# Energy fields are defined by the UCI data dictionary.
energy = energy_raw.copy()
energy["timestamp"] = pd.to_datetime(energy["date"], errors="coerce")
energy = energy.dropna(subset=["timestamp"]).sort_values("timestamp").set_index("timestamp")
energy["electricity_wh"] = pd.to_numeric(energy["Appliances"], errors="coerce") + pd.to_numeric(energy["lights"], errors="coerce")
energy = energy[["electricity_wh"] + [c for c in ["T_out", "RH_out", "Windspeed", "Visibility"] if c in energy.columns]]
energy = energy.resample("1h").mean().interpolate(limit=2)
energy = energy.dropna(subset=["electricity_wh"])
energy["electricity_kwh"] = energy["electricity_wh"] / 1000.0
print("Hourly real energy observations:", energy.shape)
display(energy.head())

In [ ]:
# This public source is annual, not daily. Preserve its actual annual resolution.
water_date_col = find_col(water_raw.columns, [r"^year$"])
water_value_col = find_col(water_raw.columns, [r"nyc_consumption", r"consumption"])
if water_date_col is None or water_value_col is None:
    raise ValueError(
        "Could not identify year and city water-consumption columns. "
        f"Available columns: {water_raw.columns.tolist()}. Update the mapping explicitly."
    )
water = water_raw[[water_date_col, water_value_col]].copy()
water.columns = ["year", "water_city_mgd"]
water["year"] = pd.to_numeric(water["year"], errors="coerce")
water["water_city_mgd"] = pd.to_numeric(water["water_city_mgd"], errors="coerce")
water = water.dropna().sort_values("year")
water["year"] = water["year"].astype(int)
water = water.drop_duplicates("year").set_index("year")
print("Annual NYC water observations:", water.shape)
print("Source field:", water_value_col, "(million gallons per day)")
display(water.head())

In [ ]:
# Waste workbook schema is intentionally inspected before mapping.
# Detect plausible waste-generation-per-capita fields; require a user-reviewed match.
waste_candidates = [c for c in waste_raw.columns if re.search(
    r"waste.*(capita|person|generation)|generation.*(capita|person)", str(c), re.I
)]
print("Candidate waste-intensity columns:", waste_candidates)
print("Waste workbook columns:", waste_raw.columns.tolist())
if not waste_candidates:
    raise ValueError(
        "No obvious per-capita waste-generation field was found. Inspect the printed "
        "World Bank workbook columns and set WASTE_VALUE_COL to the exact documented "
        "field. Do not choose a projection or unrelated indicator."
    )
WASTE_VALUE_COL = waste_candidates[0]
country_col = find_col(waste_raw.columns, [r"country.*name", r"country"])
waste_values = pd.to_numeric(waste_raw[WASTE_VALUE_COL], errors="coerce").dropna()
if waste_values.empty:
    raise ValueError("Detected waste field contains no numeric observations.")
# Use the median of observed country values as a transparent reference intensity.
# This is a global benchmark, not a college-specific measurement.
waste_intensity_kg_person_day = float(waste_values.median())
print("Selected field:", WASTE_VALUE_COL)
print("Global median waste intensity (kg/person/day):", waste_intensity_kg_person_day)

## Stage 2 — ERDP: Event-Aware Robust Data Preparation

The public time series are cleaned first. Event-level rows are then created by applying clearly declared event scenarios (attendance, duration, event category and service minima) to measured resource baselines. These rows are **simulation instances**, not observed college events.

In [ ]:
# KNN imputation, IQR flagging and robust scaling on measured energy features.
numeric_cols = energy.select_dtypes(include=np.number).columns.tolist()
imputer = KNNImputer(n_neighbors=5, weights="distance")
energy_imputed = pd.DataFrame(imputer.fit_transform(energy[numeric_cols]),
                              index=energy.index, columns=numeric_cols)
resource_cols = ["electricity_kwh"]
q1 = energy_imputed[resource_cols].quantile(0.25)
q3 = energy_imputed[resource_cols].quantile(0.75)
iqr = q3 - q1
energy_imputed["energy_iqr_outlier"] = (
    (energy_imputed["electricity_kwh"] < q1["electricity_kwh"] - 1.5 * iqr["electricity_kwh"]) |
    (energy_imputed["electricity_kwh"] > q3["electricity_kwh"] + 1.5 * iqr["electricity_kwh"])
)
energy_imputed["electricity_kwh_clean"] = energy_imputed["electricity_kwh"].clip(
    lower=max(0, q1["electricity_kwh"] - 1.5 * iqr["electricity_kwh"]),
    upper=q3["electricity_kwh"] + 1.5 * iqr["electricity_kwh"]
)
resource_scaler = RobustScaler()
energy_imputed["electricity_robust"] = resource_scaler.fit_transform(
    energy_imputed[["electricity_kwh_clean"]]
).ravel()
print("IQR outlier fraction:", energy_imputed["energy_iqr_outlier"].mean())
display(energy_imputed.head())

In [ ]:
# Explicit event scenarios. These are experimental design inputs, not historical event logs.
EVENT_SCENARIOS = pd.DataFrame([
    {"event_type":"lecture","attendance":120,"duration_hours":2.0,"electricity_factor":1.00,"water_l_per_person":2.0,"waste_kg_per_person":0.12,"min_service":0.70},
    {"event_type":"seminar","attendance":180,"duration_hours":3.0,"electricity_factor":1.15,"water_l_per_person":2.5,"waste_kg_per_person":0.16,"min_service":0.72},
    {"event_type":"sports","attendance":300,"duration_hours":4.0,"electricity_factor":0.90,"water_l_per_person":5.0,"waste_kg_per_person":0.20,"min_service":0.80},
    {"event_type":"cultural","attendance":450,"duration_hours":5.0,"electricity_factor":1.35,"water_l_per_person":3.5,"waste_kg_per_person":0.24,"min_service":0.78},
    {"event_type":"conference","attendance":250,"duration_hours":6.0,"electricity_factor":1.25,"water_l_per_person":3.0,"waste_kg_per_person":0.18,"min_service":0.75},
])
print("Scenario definitions (editable):")
display(EVENT_SCENARIOS)

In [ ]:
energy_daily = energy_imputed["electricity_kwh_clean"].resample("D").mean().dropna()
base = pd.DataFrame({"date": energy_daily.index, "electricity_kwh_baseline": energy_daily.values})
base["year"] = base["date"].dt.year
# Join annual measured city water data by calendar year; do not interpolate annual data to fake daily readings.
base = base.merge(water.reset_index(), on="year", how="inner")
if base.empty:
    raise ValueError("No calendar-year overlap between the UCI energy and NYC annual water datasets.")
water_median = max(float(base["water_city_mgd"].median()), 1e-9)
base["water_context_ratio"] = base["water_city_mgd"] / water_median
display(base.head())
records = []
for i, row in base.reset_index(drop=True).iterrows():
    sc = EVENT_SCENARIOS.iloc[i % len(EVENT_SCENARIOS)]
    e_kwh = max(0.0, row.electricity_kwh_baseline * sc.electricity_factor * sc.duration_hours)
    water_l = max(0.0, sc.attendance * sc.water_l_per_person * row.water_context_ratio)
    waste_kg = max(0.0, sc.attendance * sc.waste_kg_per_person)
    records.append({
        "date":row.date, "event_type":sc.event_type, "attendance":sc.attendance,
        "duration_hours":sc.duration_hours, "min_service":sc.min_service,
        "electricity_kwh":e_kwh, "water_litres":water_l, "waste_kg":waste_kg,
        "water_context_ratio":row.water_context_ratio,
        "energy_source":"UCI measured hourly energy aggregated to day + declared scenario multiplier",
        "water_source":"NYC measured annual city consumption contextualized by calendar year + declared per-person event coefficient",
        "waste_source":"World Bank observed country-level median intensity + declared event scenario coefficient"
    })
events = pd.DataFrame(records).sort_values("date").reset_index(drop=True)
GRID_KG_CO2_PER_KWH = 0.70
events["carbon_kg_co2e"] = events["electricity_kwh"] * GRID_KG_CO2_PER_KWH
display(events.head())
print("Event scenario rows:", events.shape)

In [ ]:
# Robust scaling of resource features; preserve physical-unit columns for reporting.
MODEL_FEATURES = ["attendance","duration_hours","water_context_ratio","electricity_kwh","water_litres","waste_kg"]
TARGETS = ["electricity_kwh","water_litres","waste_kg","carbon_kg_co2e"]
event_scaler = RobustScaler()
events_scaled = events.copy()
events_scaled[MODEL_FEATURES] = event_scaler.fit_transform(events[MODEL_FEATURES])
events.to_csv(OUTPUT_DIR / "event_oriented_sustainability_dataset.csv", index=False)
print("Saved:", OUTPUT_DIR / "event_oriented_sustainability_dataset.csv")

## Stage 3 — MEAD: MQTT-enabled edge anomaly detection

The notebook supports two modes:
- **Replay mode (default):** actual public-source observations are replayed through an in-process MQTT-like message queue so the notebook remains executable without a broker.
- **Broker mode:** set `MQTT_ENABLED=True` and provide a reachable broker to publish/subscribe using MQTT. Broker availability is external and not guaranteed.

EMA smoothing, duplicate/invalid filtering and Isolation Forest are applied in both modes. Replay mode does not claim to be a physical sensor deployment.

In [ ]:
MQTT_ENABLED = False
MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "research/college_event/resources"

sensor_stream = events[["date","electricity_kwh","water_litres","waste_kg","carbon_kg_co2e"]].copy()
sensor_stream["timestamp"] = pd.to_datetime(sensor_stream["date"])
sensor_stream = sensor_stream.drop(columns="date").to_dict("records")

if MQTT_ENABLED:
    import paho.mqtt.client as mqtt
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.connect(MQTT_BROKER, MQTT_PORT, keepalive=30)
    client.loop_start()
    for msg in sensor_stream[:500]:
        payload = json.dumps({k:(v.isoformat() if hasattr(v,"isoformat") else float(v)) for k,v in msg.items()})
        client.publish(MQTT_TOPIC, payload, qos=1)
    client.loop_stop(); client.disconnect()
    print("Published first 500 replay observations to MQTT topic:", MQTT_TOPIC)
else:
    # Local deterministic replay queue. No fabricated sensor readings are added.
    mqtt_queue = [json.dumps({k:(v.isoformat() if hasattr(v,"isoformat") else float(v)) for k,v in m.items()}) for m in sensor_stream]
    print("Local replay messages:", len(mqtt_queue))

In [ ]:
edge = pd.DataFrame(sensor_stream)
edge["timestamp"] = pd.to_datetime(edge["timestamp"])
edge = edge.drop_duplicates(subset=["timestamp"]).sort_values("timestamp")
for c in TARGETS:
    edge[c] = pd.to_numeric(edge[c], errors="coerce")
edge = edge.replace([np.inf,-np.inf], np.nan).dropna(subset=TARGETS)
for c in TARGETS:
    edge[c+"_ema"] = edge[c].ewm(span=5, adjust=False).mean()
iso = IsolationForest(n_estimators=200, contamination=0.03, random_state=SEED)
edge["anomaly"] = iso.fit_predict(edge[[c+"_ema" for c in TARGETS]])
edge["is_anomaly"] = edge["anomaly"] == -1
validated = edge.loc[~edge["is_anomaly"]].copy()
if len(validated) < 100:
    warnings.warn("Few non-anomalous points remain; training will use all validated rows.")
print("Input:", len(edge), "Anomalies:", int(edge.is_anomaly.sum()), "Validated:", len(validated))
display(edge.head())

## Stage 4 — LMSTCN: temporal multi-resource demand prediction

A compact causal/dilated depthwise-separable temporal convolutional network predicts the four resource targets. The split is chronological to avoid look-ahead leakage. The model uses event-level scenario records derived in Stage 2.

In [ ]:
class DepthwiseTemporalBlock(nn.Module):
    def __init__(self, channels, dilation, kernel=3, dropout=0.10):
        super().__init__()
        self.pad = (kernel-1)*dilation
        self.depth = nn.Conv1d(channels, channels, kernel, padding=0, dilation=dilation, groups=channels)
        self.point = nn.Conv1d(channels, channels, 1)
        self.norm = nn.BatchNorm1d(channels)
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = nn.functional.pad(x, (self.pad,0))
        y = self.depth(y)
        y = self.point(y)
        y = self.drop(self.act(self.norm(y)))
        return x + y

class LMSTCN(nn.Module):
    def __init__(self, n_features, n_targets, channels=48):
        super().__init__()
        self.proj = nn.Conv1d(n_features, channels, 1)
        self.blocks = nn.Sequential(
            DepthwiseTemporalBlock(channels,1),
            DepthwiseTemporalBlock(channels,2),
            DepthwiseTemporalBlock(channels,4),
            DepthwiseTemporalBlock(channels,8))
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(channels,n_targets))
    def forward(self,x):
        return self.head(self.blocks(self.proj(x)))

FEATURES = ["attendance","duration_hours","water_context_ratio"]
X_raw = events[FEATURES].to_numpy(dtype=np.float32)
Y_raw = events[TARGETS].to_numpy(dtype=np.float32)
x_scaler = RobustScaler().fit(X_raw)
y_scaler = RobustScaler().fit(Y_raw)
X = x_scaler.transform(X_raw).astype(np.float32)
Y = y_scaler.transform(Y_raw).astype(np.float32)
WINDOW = 24
def make_windows(x,y,window):
    xx, yy = [], []
    for i in range(window, len(x)):
        xx.append(x[i-window:i].T)
        yy.append(y[i])
    return np.asarray(xx,dtype=np.float32), np.asarray(yy,dtype=np.float32)
Xw, Yw = make_windows(X,Y,WINDOW)
split = int(len(Xw)*0.80)
X_train, X_test, y_train, y_test = Xw[:split], Xw[split:], Yw[:split], Yw[split:]
train_loader = DataLoader(TensorDataset(torch.tensor(X_train),torch.tensor(y_train)), batch_size=64, shuffle=True)
model = LMSTCN(len(FEATURES),len(TARGETS)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
loss_fn = nn.SmoothL1Loss()
EPOCHS_TCN = 30
history = []
for epoch in range(EPOCHS_TCN):
    model.train(); losses=[]
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad()
        loss=loss_fn(model(xb),yb)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        losses.append(loss.item())
    history.append(float(np.mean(losses)))
    if (epoch+1)%5==0: print(f"Epoch {epoch+1:02d}/{EPOCHS_TCN} | loss={history[-1]:.5f}")
plt.figure(figsize=(7,4)); plt.plot(history,linewidth=2); plt.xlabel("Epoch"); plt.ylabel("Training Smooth-L1 loss"); plt.title("LMSTCN training"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"lmstcn_training.png",dpi=300); plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    pred_scaled = model(torch.tensor(X_test).to(DEVICE)).cpu().numpy()
pred = y_scaler.inverse_transform(pred_scaled)
actual = y_scaler.inverse_transform(y_test)
prediction_metrics = []
for j,name in enumerate(TARGETS):
    prediction_metrics.append({"resource":name,
        "MAE":mean_absolute_error(actual[:,j],pred[:,j]),
        "RMSE":mean_squared_error(actual[:,j],pred[:,j])**0.5,
        "R2":r2_score(actual[:,j],pred[:,j])})
prediction_metrics = pd.DataFrame(prediction_metrics)
display(prediction_metrics)
prediction_metrics.to_excel(OUTPUT_DIR/"lmstcn_prediction_metrics.xlsx",index=False)

## Stage 5 — EAP-DTS: event-aware predictive digital twin synchronization

A discrete-event calendar represents setup, active event and teardown. A linear state-space estimator updates the virtual resource state from validated observations using a Kalman filter. The twin tracks electricity, water, waste and carbon state estimates.

In [ ]:
# Discrete-event representation of one event lifecycle.
def build_event_schedule(event_row):
    start = pd.Timestamp(event_row["date"])
    duration = pd.Timedelta(hours=float(event_row["duration_hours"]))
    return pd.DataFrame([
        {"time":start,"phase":"setup","fraction":0.10},
        {"time":start+duration*0.10,"phase":"active","fraction":0.80},
        {"time":start+duration*0.90,"phase":"teardown","fraction":0.10},
        {"time":start+duration,"phase":"complete","fraction":0.00},
    ])
display(build_event_schedule(events.iloc[0]))

In [ ]:
class ResourceKalman:
    def __init__(self, n=4):
        self.x = np.zeros(n, dtype=float)
        self.P = np.eye(n)*10.0
        self.Q = np.eye(n)*0.05
        self.R = np.eye(n)*0.50
        self.H = np.eye(n)
    def update(self, observation):
        z = np.asarray(observation,dtype=float)
        self.P = self.P + self.Q
        K = self.P @ self.H.T @ np.linalg.inv(self.H @ self.P @ self.H.T + self.R)
        self.x = self.x + K @ (z - self.H @ self.x)
        self.P = (np.eye(len(self.x))-K@self.H)@self.P
        return self.x.copy()

kf = ResourceKalman(len(TARGETS))
twin_rows=[]
for _,r in validated.iterrows():
    estimate=kf.update([r[c] for c in TARGETS])
    twin_rows.append({"timestamp":r.timestamp,**{f"twin_{c}":estimate[i] for i,c in enumerate(TARGETS)}})
twin = pd.DataFrame(twin_rows)
display(twin.head())
twin.to_csv(OUTPUT_DIR/"digital_twin_state_history.csv",index=False)

## Stage 6 — MO-PPO: multi-objective adaptive resource optimization

The custom PPO implementation uses an actor-critic policy with a clipped surrogate objective and GAE. The environment allocates normalized electricity, water, waste-service and reusable-material capacity. Reward terms penalize resource use, carbon, unmet demand and operational constraint violations. Minimum service constraints are enforced by a projection step.

In [ ]:
# Build an event optimization environment from predicted demands (real-data-derived scenario rows).
# Demands are normalized for stable policy learning.
demand = events[TARGETS].to_numpy(dtype=np.float32)
demand_scale = np.maximum(np.median(demand,axis=0),1e-6)
demand_norm = np.clip(demand/demand_scale,0.05,5.0)
service_min = events["min_service"].to_numpy(dtype=np.float32)

class EventResourceEnv:
    def __init__(self, demands, minimums):
        self.demands=demands; self.minimums=minimums; self.n=len(demands); self.reset()
    def reset(self):
        self.i=0
        return self._state()
    def _state(self):
        return np.r_[self.demands[self.i], self.minimums[self.i]].astype(np.float32)
    def step(self, action):
        d=self.demands[self.i]
        minimum=self.minimums[self.i]
        # First four action dimensions allocate resource coverage; fifth controls reuse effort.
        allocation=np.clip(action[:4],minimum,1.25)
        reuse=float(np.clip(action[4],0,1))
        unmet=np.maximum(d-allocation*d,0)
        normalized_use=np.mean(allocation)
        carbon_penalty=allocation[0]*d[3]
        waste_penalty=allocation[2]*d[2]*(1-0.45*reuse)
        service_penalty=np.mean(np.maximum(minimum-allocation,0))*10
        reward=-(0.25*normalized_use+0.20*carbon_penalty+0.15*waste_penalty+
                 2.0*np.mean(unmet)+service_penalty-0.10*reuse)
        self.i+=1
        done=self.i>=self.n
        nxt=np.zeros(5,dtype=np.float32) if done else self._state()
        return nxt,float(reward),done,{"allocation":allocation,"reuse":reuse,"unmet":unmet}

class ActorCritic(nn.Module):
    def __init__(self,state_dim=5,action_dim=5):
        super().__init__()
        self.body=nn.Sequential(nn.Linear(state_dim,128),nn.Tanh(),nn.Linear(128,128),nn.Tanh())
        self.mu=nn.Linear(128,action_dim)
        self.log_std=nn.Parameter(torch.full((action_dim,),-0.5))
        self.value=nn.Linear(128,1)
    def forward(self,s):
        h=self.body(s)
        return torch.sigmoid(self.mu(h))*1.25, self.log_std.expand_as(self.mu(h)), self.value(h).squeeze(-1)
    def act(self,s):
        mu,log_std,v=self(s)
        dist=torch.distributions.Normal(mu,log_std.exp())
        z=dist.rsample()
        a=torch.sigmoid(z)*1.25
        logp=dist.log_prob(z).sum(-1)
        return a,logp,v

def train_moppo(demands, minimums, updates=250, gamma=0.99, lam=0.95, clip=0.2):
    env=EventResourceEnv(demands,minimums)
    agent=ActorCritic().to(DEVICE)
    opt=torch.optim.Adam(agent.parameters(),lr=3e-4)
    reward_history=[]
    for update in range(updates):
        states=[]; actions=[]; logps=[]; rewards=[]; dones=[]; values=[]
        s=env.reset(); ep_reward=0
        while True:
            st=torch.tensor(s,dtype=torch.float32,device=DEVICE).unsqueeze(0)
            with torch.no_grad(): a,lp,v=agent.act(st)
            action=a.squeeze(0).cpu().numpy()
            ns,r,done,info=env.step(action)
            states.append(s); actions.append(action); logps.append(float(lp.item()))
            rewards.append(r); dones.append(float(done)); values.append(float(v.item()))
            ep_reward+=r; s=ns
            if done: break
        with torch.no_grad():
            last_v=0.0
        adv=np.zeros(len(rewards),dtype=np.float32); gae=0.0
        for t in reversed(range(len(rewards))):
            next_v=last_v if t==len(rewards)-1 else values[t+1]
            delta=rewards[t]+gamma*next_v*(1-dones[t])-values[t]
            gae=delta+gamma*lam*(1-dones[t])*gae
            adv[t]=gae
        returns=adv+np.asarray(values,dtype=np.float32)
        st=torch.tensor(np.asarray(states),dtype=torch.float32,device=DEVICE)
        at=torch.tensor(np.asarray(actions),dtype=torch.float32,device=DEVICE)
        oldlp=torch.tensor(np.asarray(logps),dtype=torch.float32,device=DEVICE)
        advt=torch.tensor(adv,dtype=torch.float32,device=DEVICE)
        rett=torch.tensor(returns,dtype=torch.float32,device=DEVICE)
        advt=(advt-advt.mean())/(advt.std()+1e-8)
        for _ in range(4):
            mu,log_std,val=agent(st)
            # Inverse of bounded sigmoid action for a stable Gaussian log probability.
            z=torch.logit(torch.clamp(at/1.25,1e-5,1-1e-5))
            dist=torch.distributions.Normal(mu,log_std.exp())
            newlp=dist.log_prob(z).sum(-1)
            ratio=torch.exp(newlp-oldlp)
            surr1=ratio*advt
            surr2=torch.clamp(ratio,1-clip,1+clip)*advt
            loss=-(torch.minimum(surr1,surr2)).mean()+0.5*((val-rett)**2).mean()-0.01*dist.entropy().sum(-1).mean()
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(agent.parameters(),1.0); opt.step()
        reward_history.append(ep_reward)
        if (update+1)%50==0: print(f"PPO update {update+1}/{updates} | episode reward {ep_reward:.4f}")
    return agent,reward_history

ppo_agent, ppo_rewards = train_moppo(demand_norm,service_min,updates=250)
plt.figure(figsize=(7,4)); plt.plot(ppo_rewards); plt.xlabel("PPO update"); plt.ylabel("Episode reward"); plt.title("MO-PPO learning curve"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"moppo_learning_curve.png",dpi=300); plt.show()

In [ ]:
# Deterministic policy rollout and service-constraint projection.
allocations=[]
ppo_agent.eval()
with torch.no_grad():
    for i in range(len(demand_norm)):
        state=torch.tensor(np.r_[demand_norm[i],service_min[i]],dtype=torch.float32,device=DEVICE).unsqueeze(0)
        mu,_,_=ppo_agent(state)
        a=mu.squeeze(0).cpu().numpy()
        a[:4]=np.maximum(a[:4],service_min[i])
        allocations.append(a)
allocations=np.asarray(allocations)
allocation_df=pd.DataFrame(allocations,columns=["electricity_fraction","water_fraction","waste_service_fraction","carbon_budget_fraction","reuse_intensity"])
allocation_df.to_csv(OUTPUT_DIR/"moppo_resource_allocations.csv",index=False)
display(allocation_df.head())

## Stage 7 — SA-ARA: sustainability-aware adaptive resource allocation

Translate policy outputs into operational recommendations. This is a decision-support simulation: the notebook does not actuate campus equipment, open valves, or dispatch vehicles. Waste collection timing is represented as a service-priority score; reusable-material allocation is represented by the learned reuse intensity.

In [ ]:
green = events[["date","event_type","attendance","duration_hours"]].copy()
green["electricity_allocation_kwh"] = events["electricity_kwh"]*allocation_df["electricity_fraction"].to_numpy()
green["water_allocation_litres"] = events["water_litres"]*allocation_df["water_fraction"].to_numpy()
green["waste_collection_capacity_kg"] = events["waste_kg"]*allocation_df["waste_service_fraction"].to_numpy()
green["reuse_intensity"] = allocation_df["reuse_intensity"].to_numpy()
green["estimated_carbon_kg"] = green["electricity_allocation_kwh"]*GRID_KG_CO2_PER_KWH
green["collection_priority"] = (
    green["waste_collection_capacity_kg"] / green["waste_collection_capacity_kg"].median()
).clip(0,2)
green["reusable_material_units_index"] = green["attendance"]*green["reuse_intensity"]
green["water_saving_vs_baseline_l"] = (events["water_litres"]-green["water_allocation_litres"]).clip(lower=0)
green["energy_saving_vs_baseline_kwh"] = (events["electricity_kwh"]-green["electricity_allocation_kwh"]).clip(lower=0)
display(green.head())
green.to_excel(OUTPUT_DIR/"sa_ara_event_recommendations.xlsx",index=False)

## Stage 8 — EW-ESA: entropy-weighted sustainability assessment

Entropy weights are calculated from non-negative benefit/cost indicators. TOPSIS ranks event records, and the Event Sustainability Index (ESI) is reported on a 0–100 scale. Carbon is derived from electricity using the explicit grid-emission factor configured above. Results are comparative scenario scores, not externally audited sustainability certifications.

In [ ]:
# Criteria matrix: resource savings/reuse are benefits; remaining resource use/carbon are costs.
criteria = pd.DataFrame({
    "energy_efficiency": green["energy_saving_vs_baseline_kwh"].to_numpy(),
    "water_efficiency": green["water_saving_vs_baseline_l"].to_numpy(),
    "waste_service": green["waste_collection_capacity_kg"].to_numpy(),
    "reuse": green["reuse_intensity"].to_numpy(),
    "carbon_cost": green["estimated_carbon_kg"].to_numpy()
})
benefit = np.array([True,True,True,True,False])
Xcrit=criteria.to_numpy(dtype=float)
# Shift cost/benefit data to nonnegative proportions for entropy computation.
P=np.zeros_like(Xcrit,dtype=float)
for j in range(Xcrit.shape[1]):
    col=Xcrit[:,j].copy()
    if benefit[j]:
        col=col-col.min()+1e-9
    else:
        col=col.max()-col+1e-9
    P[:,j]=col/(col.sum()+1e-12)
n=len(P); k=1/np.log(max(n,2))
entropy=-k*np.sum(P*np.log(P+1e-12),axis=0)
divergence=1-entropy
weights=divergence/(divergence.sum()+1e-12)
weight_table=pd.DataFrame({"criterion":criteria.columns,"entropy":entropy,"weight":weights,"benefit":benefit})
display(weight_table)
# TOPSIS vector normalization and weighted ideal distances.
norm=Xcrit/(np.sqrt((Xcrit**2).sum(axis=0))+1e-12)
weighted=norm*weights
ideal_best=np.where(benefit,weighted.max(axis=0),weighted.min(axis=0))
ideal_worst=np.where(benefit,weighted.min(axis=0),weighted.max(axis=0))
d_best=np.sqrt(((weighted-ideal_best)**2).sum(axis=1))
d_worst=np.sqrt(((weighted-ideal_worst)**2).sum(axis=1))
closeness=d_worst/(d_best+d_worst+1e-12)
green["topsis_score"]=closeness
green["ESI_0_100"]=100*closeness
green["sustainability_rank"]=green["ESI_0_100"].rank(ascending=False,method="min").astype(int)
display(green.sort_values("sustainability_rank").head(10))
weight_table.to_excel(OUTPUT_DIR/"ewm_entropy_weights.xlsx",index=False)
green.to_excel(OUTPUT_DIR/"ew_esa_sustainability_assessment.xlsx",index=False)

In [ ]:
# Aggregate sustainability and resource-efficiency indicators.
assessment = pd.DataFrame([{
    "events_evaluated":len(green),
    "mean_ESI":green["ESI_0_100"].mean(),
    "median_ESI":green["ESI_0_100"].median(),
    "energy_savings_kwh":green["energy_saving_vs_baseline_kwh"].sum(),
    "water_savings_litres":green["water_saving_vs_baseline_l"].sum(),
    "estimated_carbon_kg_co2e":green["estimated_carbon_kg"].sum(),
    "mean_reuse_intensity":green["reuse_intensity"].mean(),
    "mean_waste_collection_capacity_kg":green["waste_collection_capacity_kg"].mean()
}])
display(assessment)
assessment.to_excel(OUTPUT_DIR/"overall_sustainability_assessment.xlsx",index=False)
plt.figure(figsize=(8,4)); plt.hist(green["ESI_0_100"],bins=20); plt.xlabel("Event Sustainability Index"); plt.ylabel("Number of scenario events"); plt.title("Distribution of event sustainability scores"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"esi_distribution.png",dpi=300); plt.show()
plt.figure(figsize=(8,4)); plt.plot(green["date"],green["ESI_0_100"],linewidth=1.5); plt.xlabel("Date index from public baseline"); plt.ylabel("ESI (0–100)"); plt.title("Scenario event sustainability over time"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"esi_timeline.png",dpi=300); plt.show()

## Baseline comparison and statistical evaluation

The comparison below is a transparent heuristic baseline, not a competing published algorithm. It allocates all resources at 100% of demand and uses no learned reuse policy. A paired Wilcoxon test compares event-level ESI values. Interpret the p-value cautiously because the events are constructed scenarios and are not independent real-world college events.

In [ ]:
baseline_green=events.copy()
baseline_criteria=pd.DataFrame({
    "energy_efficiency":np.zeros(len(events)),
    "water_efficiency":np.zeros(len(events)),
    "waste_service":events["waste_kg"].to_numpy(),
    "reuse":np.zeros(len(events)),
    "carbon_cost":events["carbon_kg_co2e"].to_numpy()
})
# Same EWM/TOPSIS routine applied to baseline.
def topsis_scores(frame, benefit_flags):
    a=frame.to_numpy(float); b=np.asarray(benefit_flags,bool)
    p=np.zeros_like(a)
    for j in range(a.shape[1]):
        c=a[:,j].copy()
        c=c-c.min()+1e-9 if b[j] else c.max()-c+1e-9
        p[:,j]=c/(c.sum()+1e-12)
    ent=-(1/np.log(max(len(a),2)))*np.sum(p*np.log(p+1e-12),axis=0)
    w=(1-ent)/(np.sum(1-ent)+1e-12)
    z=a/(np.sqrt((a*a).sum(axis=0))+1e-12)*w
    bp=np.where(b,z.max(0),z.min(0)); bn=np.where(b,z.min(0),z.max(0))
    dp=np.sqrt(((z-bp)**2).sum(1)); dn=np.sqrt(((z-bn)**2).sum(1))
    return dn/(dp+dn+1e-12)
baseline_esi=100*topsis_scores(baseline_criteria,[True,True,True,True,False])
proposed_esi=green["ESI_0_100"].to_numpy()
comparison=pd.DataFrame({
    "method":["Full-demand heuristic baseline","MO-PPO + SA-ARA"],
    "mean_ESI":[baseline_esi.mean(),proposed_esi.mean()],
    "median_ESI":[np.median(baseline_esi),np.median(proposed_esi)]
})
display(comparison)
try:
    stat,pvalue=wilcoxon(proposed_esi,baseline_esi,alternative="greater")
except ValueError:
    stat,pvalue=np.nan,np.nan
print("Paired Wilcoxon statistic:",stat,"p-value:",pvalue)
comparison.to_excel(OUTPUT_DIR/"method_comparison.xlsx",index=False)

## Export manifest and reproducibility record

In [ ]:
manifest = {
    "seed":SEED,
    "device":str(DEVICE),
    "output_directory":str(OUTPUT_DIR.resolve()),
    "sources":{
        "energy":"UCI Appliances Energy Prediction, https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction",
        "water":"NYC Open Data Socrata endpoint, "+NYC_WATER_URL,
        "waste":"World Bank What a Waste 3.0, https://datacatalog.worldbank.org/search/dataset/0039597/what-a-waste-global-database"
    },
    "scenario_notice":"Event attendance, duration, event type, service minima and per-person resource coefficients are explicit simulation inputs, not observed historical college events.",
    "grid_emission_factor_kg_co2e_per_kwh":GRID_KG_CO2_PER_KWH,
    "files":sorted(p.name for p in OUTPUT_DIR.glob("*") if p.is_file())
}
(OUTPUT_DIR/"run_manifest.json").write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
print("\nNotebook pipeline completed. Outputs are stored in:",OUTPUT_DIR.resolve())

## Research integrity and reporting notes

- Do not describe the scenario rows as real college event records. They are event-level experimental scenarios constructed from public measurements and explicitly stated assumptions.
- The UCI energy dataset is residential/building data, and NYC water data is municipal/city-level. They are not campus-meter readings. A real campus deployment requires campus submeters, water meters, waste weighing records, event attendance logs, and event schedules.
- The World Bank waste indicator is a national/city benchmark. Its median is used as a documented reference intensity; the chosen column and workbook sheet are printed in the notebook for audit.
- The MQTT default is local replay. To demonstrate actual broker transport, configure a reachable broker and set `MQTT_ENABLED=True`. This does not establish secure production IoT communication; production deployment requires authentication, TLS, topic ACLs and device identity.
- The PPO environment is a research simulator. Before operational use, calibrate reward coefficients, resource constraints, equipment limits, and safety interlocks with campus facilities staff.
- Cite the original datasets and report the scenario assumptions, source limitations, and emission factor in any publication.